# Задание 5. Извлечение частотности слов с помощью модели Bag of Words

Чанки корпуса Brown (из задания 4) рассматриваются как документы. Каждый чанк получает индекс, затем строится терм-документная матрица с помощью `CountVectorizer`.

In [1]:
import nltk
from nltk.corpus import brown
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

nltk.download("brown", quiet=True)

[nltk_data] Error loading brown: Security Violation [pathsec.urlopen]:
[nltk_data]     refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.


False

In [2]:
N = 1000  # количество первых слов из корпуса Brown (не менее 1000)
M = 20    # количество слов в одном чанке

words = brown.words()[:N]
chunks = [words[i:i + M] for i in range(0, len(words), M)]

# Элементы словаря: индекс чанка -> текст чанка
docs = {i: " ".join(chunk) for i, chunk in enumerate(chunks)}
corpus = list(docs.values())

print("Количество чанков:", len(docs))
print("Пример чанка 0:", docs[0][:80])

Количество чанков: 50
Пример чанка 0: The Fulton County Grand Jury said Friday an investigation of Atlanta's recent pr


In [3]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

terms = vectorizer.get_feature_names_out()
matrix = X.toarray()

print("Размер матрицы (документы x термы):", matrix.shape)
print("Размер словаря:", len(terms))

# Фрагмент матрицы: первые 5 чанков, первые 12 терминов
n_docs, n_terms = 5, 12
print("\n" + "doc".ljust(6) + "".join(t[:10].rjust(11) for t in terms[:n_terms]))
for i in range(n_docs):
    print(str(i).ljust(6) + "".join(str(v).rjust(11) for v in matrix[i, :n_terms]))

Размер матрицы (документы x термы): (50, 400)
Размер словаря: 400

doc            13       1913       1923       1937       1962        637         71         74   accepted    achieve        act    actions
0               0          0          0          0          0          0          0          0          0          0          0          0
1               0          0          0          0          0          0          0          0          0          0          0          0
2               0          0          0          0          0          0          0          0          0          0          0          0
3               0          0          0          0          0          0          0          0          0          0          0          0
4               0          0          0          0          0          0          0          0          0          0          0          0


In [4]:
print("Сумма счетчиков по чанкам (первые 10):", matrix.sum(axis=1)[:10].tolist())
print("Максимальный счетчик термина:", int(matrix.max()))
print("Чанков, содержащих 'the':", int((matrix[:, list(terms).index("the")] > 0).sum()))

totals = matrix.sum(axis=0)
print("\nТоп-10 самых частых терминов:")
for t in totals.argsort()[::-1][:10]:
    print(f"  {terms[t]:<12}{int(totals[t])}")

print("\nДоля нулевых ячеек матрицы: {:.1f}%".format(100 * (matrix == 0).mean()))

Сумма счетчиков по чанкам (первые 10): [18, 19, 18, 20, 20, 13, 17, 19, 17, 16]
Максимальный счетчик термина: 4
Чанков, содержащих 'the': 48

Топ-10 самых частых терминов:
  the         82
  of          36
  and         27
  jury        18
  in          18
  that        16
  to          14
  fulton      14
  said        14
  for         10

Доля нулевых ячеек матрицы: 96.0%
